# 01c. Estatísticas descritivas das bases

Duas visões, **antes** do linkage:

1. **Brasil (bronze)** — Censo e CPF nacionais, independentes.
2. **UF 21 (limpo)** — recorte experimental já tratado (`00`/`00b`), usado para
   parametrizar o pipeline.

Saídas em `OUTPUT_DIR` / tabelas DuckDB para colar no relatório.


In [ ]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    CENSO_PESSOAS_ARQUIVO,
    CENSO_PES_NOME_ARQUIVO,
    CPF_ARQUIVO,
    CPF_CPF_NOME_ARQUIVO,
    LISTA_OURO_AVALIACAO_ARQUIVO,
    LISTA_OURO_ARQUIVO,
    OUTPUT_DIR,
    TABELA_CENSO_LIMPA,
    TABELA_CPF_LIMPA,
    cpf_norm_sql,
    ensure_output_dir,
    get_connection,
    lista_ouro_colunas,
    print_paths,
    require_input,
    require_tables,
)

print_paths()
ensure_output_dir()
con = get_connection()
require_input(CPF_ARQUIVO, label='CPF bronze')
require_input(CENSO_PESSOAS_ARQUIVO, label='Censo pessoas')
require_input(CENSO_PES_NOME_ARQUIVO, label='Censo nomes')
require_input(CPF_CPF_NOME_ARQUIVO, label='CPF nomes')
require_tables(con, [TABELA_CENSO_LIMPA, TABELA_CPF_LIMPA], notebook_origem='00b')


## 1. Brasil — bronze (antes do filtro de UF)

Volumes e completude nas fontes nacionais. Não é o limpo tratado;
responde à visão geral independente do linkage.


In [ ]:
cpf_path = str(CPF_ARQUIVO).replace("'", "''")
censo_path = str(CENSO_PESSOAS_ARQUIVO).replace("'", "''")
censo_nome_path = str(CENSO_PES_NOME_ARQUIVO).replace("'", "''")
cpf_nome_path = str(CPF_CPF_NOME_ARQUIVO).replace("'", "''")

# Totais e chaves distintas.
brasil_totais = con.execute(f'''
SELECT 'cpf_bronze' AS base, COUNT(*) AS n, COUNT(DISTINCT "COD_CPF") AS n_id
FROM read_parquet('{cpf_path}')
UNION ALL
SELECT 'cpf_nome', COUNT(*), COUNT(DISTINCT "COD_CPF")
FROM read_parquet('{cpf_nome_path}')
UNION ALL
SELECT 'censo_pessoas', COUNT(*), COUNT(DISTINCT CAST("B0000" AS VARCHAR))
FROM read_parquet('{censo_path}')
UNION ALL
SELECT 'censo_pes_nome', COUNT(*), COUNT(DISTINCT CAST("ID_MORADOR" AS VARCHAR))
FROM read_parquet('{censo_nome_path}')
''').df()
display(brasil_totais)

# Cobertura geográfica CPF (código município 7 dígitos).
brasil_geo_cpf = con.execute(f'''
SELECT
    CASE
        WHEN regexp_replace(CAST("COD_UFMUN" AS VARCHAR), '[^0-9]', '', 'g') = ''
            THEN NULL
        ELSE substr(
            lpad(regexp_replace(CAST("COD_UFMUN" AS VARCHAR), '[^0-9]', '', 'g'), 7, '0'),
            1, 2
        )
    END AS uf,
    COUNT(*) AS n
FROM read_parquet('{cpf_path}')
GROUP BY 1
ORDER BY n DESC
''').df()
print('UFs distintas (CPF):', brasil_geo_cpf['uf'].dropna().nunique())
display(brasil_geo_cpf.head(30))

brasil_geo_censo = con.execute(f'''
SELECT
    substr(
        CASE
            WHEN regexp_replace(CAST("B0000" AS VARCHAR), '[^0-9]', '', 'g') = ''
                THEN NULL
            ELSE lpad(
                regexp_replace(CAST("B0000" AS VARCHAR), '[^0-9]', '', 'g'),
                15, '0'
            )
        END,
        1, 2
    ) AS uf,
    COUNT(*) AS n
FROM read_parquet('{censo_path}')
GROUP BY 1
ORDER BY n DESC
''').df()
print('UFs distintas (Censo pessoas):', brasil_geo_censo['uf'].dropna().nunique())
display(brasil_geo_censo.head(30))


In [ ]:
# Completude bronze CPF (campos crus usados no linkage).
brasil_miss_cpf = con.execute(f'''
SELECT
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE "NOM_PESSOA" IS NOT NULL AND trim(CAST("NOM_PESSOA" AS VARCHAR)) <> ''
    ) / COUNT(*), 1) AS pct_nome,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE "DAT_NASCIMENTO" IS NOT NULL
          AND trim(CAST("DAT_NASCIMENTO" AS VARCHAR)) <> ''
    ) / COUNT(*), 1) AS pct_data,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE "COD_SEXO" IS NOT NULL AND trim(CAST("COD_SEXO" AS VARCHAR)) <> ''
    ) / COUNT(*), 1) AS pct_sexo,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE "NOM_MAE" IS NOT NULL AND trim(CAST("NOM_MAE" AS VARCHAR)) <> ''
    ) / COUNT(*), 1) AS pct_mae,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE "COD_UFMUN" IS NOT NULL
          AND regexp_replace(CAST("COD_UFMUN" AS VARCHAR), '[^0-9]', '', 'g') <> ''
    ) / COUNT(*), 1) AS pct_municipio,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE "COD_CEP" IS NOT NULL
          AND regexp_replace(CAST("COD_CEP" AS VARCHAR), '[^0-9]', '', 'g') <> ''
    ) / COUNT(*), 1) AS pct_cep,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE "NOM_LOGRADOURO" IS NOT NULL
          AND trim(CAST("NOM_LOGRADOURO" AS VARCHAR)) <> ''
    ) / COUNT(*), 1) AS pct_logradouro
FROM read_parquet('{cpf_path}')
''').df()
brasil_miss_cpf.insert(0, 'base', 'cpf_bronze')
display(brasil_miss_cpf)

# Censo pessoas: sexo/idade no questionário; nome vem da tabela de nomes.
brasil_miss_censo = con.execute(f'''
SELECT
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE "PECP0401" IS NOT NULL
    ) / COUNT(*), 1) AS pct_idade_calc,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE "B0000" IS NOT NULL
          AND regexp_replace(CAST("B0000" AS VARCHAR), '[^0-9]', '', 'g') <> ''
    ) / COUNT(*), 1) AS pct_id_morador
FROM read_parquet('{censo_path}')
''').df()
brasil_miss_censo.insert(0, 'base', 'censo_pessoas')
display(brasil_miss_censo)

brasil_nome_censo = con.execute(f'''
SELECT
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE pes_nome IS NOT NULL AND trim(CAST(pes_nome AS VARCHAR)) <> ''
    ) / COUNT(*), 1) AS pct_nome,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE pes_nome_fonetico IS NOT NULL
          AND trim(CAST(pes_nome_fonetico AS VARCHAR)) <> ''
    ) / COUNT(*), 1) AS pct_nome_phon
FROM read_parquet('{censo_nome_path}')
''').df()
brasil_nome_censo.insert(0, 'base', 'censo_pes_nome')
display(brasil_nome_censo)

brasil_nome_cpf = con.execute(f'''
SELECT
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE cpf_nome IS NOT NULL AND trim(CAST(cpf_nome AS VARCHAR)) <> ''
    ) / COUNT(*), 1) AS pct_nome,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE cpf_nome_fonetico IS NOT NULL
          AND trim(CAST(cpf_nome_fonetico AS VARCHAR)) <> ''
    ) / COUNT(*), 1) AS pct_nome_phon
FROM read_parquet('{cpf_nome_path}')
''').df()
brasil_nome_cpf.insert(0, 'base', 'cpf_cpf_nome')
display(brasil_nome_cpf)


In [ ]:
# Sexo e indício de duplicidade no bronze CPF.
display(con.execute(f'''
SELECT
    CAST("COD_SEXO" AS VARCHAR) AS sexo,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM read_parquet('{cpf_path}')
GROUP BY 1
ORDER BY n DESC
''').df())

display(con.execute(f'''
SELECT
    COUNT(*) AS n_linhas,
    COUNT(DISTINCT "COD_CPF") AS n_cpf_distintos,
    COUNT(*) - COUNT(DISTINCT "COD_CPF") AS n_linhas_extra
FROM read_parquet('{cpf_path}')
''').df())

display(con.execute(f'''
SELECT
    COUNT(*) AS n_linhas,
    COUNT(DISTINCT CAST("ID_MORADOR" AS VARCHAR)) AS n_id_distintos,
    COUNT(*) - COUNT(DISTINCT CAST("ID_MORADOR" AS VARCHAR)) AS n_linhas_extra
FROM read_parquet('{censo_nome_path}')
''').df())

# Tokens do nome fonético (amostra 2M para não estourar memória).
display(con.execute(f'''
WITH s AS (
    SELECT cpf_nome_fonetico AS nome_phon
    FROM read_parquet('{cpf_nome_path}')
    WHERE cpf_nome_fonetico IS NOT NULL
    USING SAMPLE 2000000 ROWS
)
SELECT
    CASE
        WHEN len(string_split(nome_phon, ' ')) <= 1 THEN '1'
        WHEN len(string_split(nome_phon, ' ')) = 2 THEN '2'
        WHEN len(string_split(nome_phon, ' ')) = 3 THEN '3'
        WHEN len(string_split(nome_phon, ' ')) = 4 THEN '4'
        ELSE '5+'
    END AS faixa_tokens,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM s
GROUP BY 1
ORDER BY faixa_tokens
''').df())

display(con.execute(f'''
WITH s AS (
    SELECT pes_nome_fonetico AS nome_phon
    FROM read_parquet('{censo_nome_path}')
    WHERE pes_nome_fonetico IS NOT NULL
    USING SAMPLE 2000000 ROWS
)
SELECT
    CASE
        WHEN len(string_split(nome_phon, ' ')) <= 1 THEN '1'
        WHEN len(string_split(nome_phon, ' ')) = 2 THEN '2'
        WHEN len(string_split(nome_phon, ' ')) = 3 THEN '3'
        WHEN len(string_split(nome_phon, ' ')) = 4 THEN '4'
        ELSE '5+'
    END AS faixa_tokens,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM s
GROUP BY 1
ORDER BY faixa_tokens
''').df())


## 2. UF 21 — limpo (recorte experimental)

Universo após `00` (filtro UF) + `00b` (limpeza). Censo e CPF lado a lado.


In [ ]:
from config import (
    CENSO_LIMPO_APLICACAO,
    CPF_LIMPO_APLICACAO,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    load_table_from_parquet,
)

load_table_from_parquet(con, TABELA_CENSO_LIMPA_APLICACAO, CENSO_LIMPO_APLICACAO)
load_table_from_parquet(con, TABELA_CPF_LIMPA_APLICACAO, CPF_LIMPO_APLICACAO)

# Funil no recorte atual (limpo + aplicação).
funil = con.execute(f'''
SELECT 'censo_limpo' AS etapa, COUNT(*) AS n FROM {TABELA_CENSO_LIMPA}
UNION ALL
SELECT 'cpf_limpo', COUNT(*) FROM {TABELA_CPF_LIMPA}
UNION ALL
SELECT 'censo_limpo_aplicacao', COUNT(*) FROM {TABELA_CENSO_LIMPA_APLICACAO}
UNION ALL
SELECT 'cpf_limpo_aplicacao', COUNT(*) FROM {TABELA_CPF_LIMPA_APLICACAO}
''').df()
display(funil)

# Remoções 00b neste recorte (sem nome; óbito desligado com corte 0).
n_censo_reg = con.execute('''
SELECT COUNT(*) FROM information_schema.tables
WHERE table_schema = 'main' AND table_name = 'censo_registros'
''').fetchone()[0]
if n_censo_reg:
    display(con.execute(f'''
    SELECT
        (SELECT COUNT(*) FROM censo_registros) AS censo_registros,
        (SELECT COUNT(*) FROM {TABELA_CENSO_LIMPA}) AS censo_limpo,
        (SELECT COUNT(*) FROM censo_registros)
            - (SELECT COUNT(*) FROM {TABELA_CENSO_LIMPA}) AS removidos_censo,
        (SELECT COUNT(*) FROM cpf_registros) AS cpf_registros,
        (SELECT COUNT(*) FROM {TABELA_CPF_LIMPA}) AS cpf_limpo,
        (SELECT COUNT(*) FROM cpf_registros)
            - (SELECT COUNT(*) FROM {TABELA_CPF_LIMPA}) AS removidos_cpf
    ''').df())


In [ ]:
# Completude limpo: Censo × CPF.
uf21_completude = con.execute(f'''
SELECT
    origem,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) FILTER (WHERE nome_completo IS NOT NULL) / COUNT(*), 1)
        AS pct_nome,
    ROUND(100.0 * COUNT(*) FILTER (WHERE data_nascimento IS NOT NULL) / COUNT(*), 1)
        AS pct_data,
    ROUND(100.0 * COUNT(*) FILTER (WHERE sexo IS NOT NULL) / COUNT(*), 1)
        AS pct_sexo,
    ROUND(100.0 * COUNT(*) FILTER (WHERE nome_mae IS NOT NULL) / COUNT(*), 1)
        AS pct_mae,
    ROUND(100.0 * COUNT(*) FILTER (WHERE cod_municipio IS NOT NULL) / COUNT(*), 1)
        AS pct_municipio,
    ROUND(100.0 * COUNT(*) FILTER (WHERE cep IS NOT NULL) / COUNT(*), 1)
        AS pct_cep,
    ROUND(100.0 * COUNT(*) FILTER (WHERE logradouro_norm IS NOT NULL) / COUNT(*), 1)
        AS pct_logradouro,
    ROUND(100.0 * COUNT(*) FILTER (WHERE nome_meio IS NOT NULL) / COUNT(*), 1)
        AS pct_nome_meio,
    ROUND(100.0 * COUNT(*) FILTER (WHERE ultimo_nome IS NOT NULL) / COUNT(*), 1)
        AS pct_ultimo
FROM (
    SELECT
        'censo' AS origem,
        nome_completo, data_nascimento, sexo, nome_mae,
        cod_municipio, cep, logradouro_norm, nome_meio, ultimo_nome
    FROM {TABELA_CENSO_LIMPA}
    UNION ALL
    SELECT
        'cpf',
        nome_completo, data_nascimento, sexo, nome_mae,
        cod_municipio, cep, logradouro_norm, nome_meio, ultimo_nome
    FROM {TABELA_CPF_LIMPA}
)
GROUP BY 1
ORDER BY 1
''').df()
display(uf21_completude)

# Sexo.
display(con.execute(f'''
SELECT origem, sexo, COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY origem), 2) AS pct
FROM (
    SELECT 'censo' AS origem, sexo FROM {TABELA_CENSO_LIMPA}
    UNION ALL
    SELECT 'cpf', sexo FROM {TABELA_CPF_LIMPA}
)
GROUP BY 1, 2
ORDER BY 1, 2
''').df())

# Idade (quando preenchida).
display(con.execute(f'''
SELECT
    origem,
    COUNT(*) AS n,
    COUNT(idade) AS com_idade,
    ROUND(100.0 * COUNT(idade) / COUNT(*), 1) AS pct_idade,
    ROUND(AVG(idade), 1) AS idade_media,
    ROUND(median(idade), 1) AS idade_mediana
FROM (
    SELECT 'censo' AS origem, idade FROM {TABELA_CENSO_LIMPA}
    UNION ALL
    SELECT 'cpf', idade FROM {TABELA_CPF_LIMPA}
)
GROUP BY 1
ORDER BY 1
''').df())


In [ ]:
# Tokens do nome fonético (limpo UF 21, universo completo).
uf21_tokens = con.execute(f'''
SELECT
    origem,
    CASE
        WHEN coalesce(len(string_split(nome_completo_phon, ' ')), 0) <= 1 THEN '1'
        WHEN len(string_split(nome_completo_phon, ' ')) = 2 THEN '2'
        WHEN len(string_split(nome_completo_phon, ' ')) = 3 THEN '3'
        WHEN len(string_split(nome_completo_phon, ' ')) = 4 THEN '4'
        ELSE '5+'
    END AS faixa_tokens,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY origem), 1) AS pct
FROM (
    SELECT 'censo' AS origem, nome_completo_phon FROM {TABELA_CENSO_LIMPA}
    UNION ALL
    SELECT 'cpf', nome_completo_phon FROM {TABELA_CPF_LIMPA}
)
GROUP BY 1, 2
ORDER BY 1, 2
''').df()
display(uf21_tokens)

# Top municípios.
display(con.execute(f'''
SELECT origem, cod_municipio, COUNT(*) AS n
FROM (
    SELECT 'censo' AS origem, cod_municipio FROM {TABELA_CENSO_LIMPA}
    UNION ALL
    SELECT 'cpf', cod_municipio FROM {TABELA_CPF_LIMPA}
)
GROUP BY 1, 2
ORDER BY n DESC
LIMIT 12
''').df())

# Duplicidade de chaves no limpo.
display(con.execute(f'''
SELECT
    'censo' AS origem,
    COUNT(*) AS n,
    COUNT(DISTINCT unique_id) AS n_unique_id,
    COUNT(*) - COUNT(DISTINCT unique_id) AS extra_unique_id,
    COUNT(DISTINCT person_id_censo) AS n_person_id
FROM {TABELA_CENSO_LIMPA}
UNION ALL
SELECT
    'cpf',
    COUNT(*),
    COUNT(DISTINCT unique_id),
    COUNT(*) - COUNT(DISTINCT unique_id),
    COUNT(DISTINCT cpf_norm)
FROM {TABELA_CPF_LIMPA}
''').df())


## 3. Conjunto de referência (lista de ouro) no limpo UF 21


In [ ]:
require_input(LISTA_OURO_AVALIACAO_ARQUIVO, label='LISTA_OURO_AVALIACAO')
ouro_path = str(LISTA_OURO_AVALIACAO_ARQUIVO).replace("'", "''")
ouro_cols = {
    r[0]
    for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{ouro_path}') LIMIT 0"
    ).fetchall()
}
id_col, cpf_col = lista_ouro_colunas(ouro_cols)
cpf_sql = cpf_norm_sql(cpf_col)

con.execute(f'''
CREATE OR REPLACE TEMP TABLE lista_ouro_aval AS
SELECT
    CAST({id_col} AS VARCHAR) AS id_censo,
    {cpf_sql} AS cpf_norm
FROM read_parquet('{ouro_path}')
''')

display(con.execute('''
SELECT
    COUNT(*) AS n_pares_lista,
    COUNT(DISTINCT id_censo) AS n_censo_lista,
    COUNT(DISTINCT cpf_norm) AS n_cpf_lista
FROM lista_ouro_aval
''').df())

# Mesmo G do 03c: 1:1 e os dois ids no limpo.
con.execute('''
CREATE OR REPLACE TEMP TABLE gt_lista AS
SELECT
    'censo_' || id_censo AS unique_id_censo,
    'cpf_' || cpf_norm AS unique_id_cpf,
    id_censo,
    cpf_norm
FROM lista_ouro_aval
QUALIFY
    COUNT(*) OVER (PARTITION BY id_censo) = 1
    AND COUNT(*) OVER (PARTITION BY cpf_norm) = 1
''')

display(con.execute(f'''
SELECT
    (SELECT COUNT(*) FROM lista_ouro_aval) AS n_lista,
    (SELECT COUNT(*) FROM gt_lista) AS n_1a1,
    (
        SELECT COUNT(*)
        FROM gt_lista g
        JOIN {TABELA_CENSO_LIMPA} c ON c.unique_id = g.unique_id_censo
        JOIN {TABELA_CPF_LIMPA} p ON p.unique_id = g.unique_id_cpf
    ) AS n_ouro_no_limpo
''').df())

print('LISTA_OURO (exclusão aplicação):', LISTA_OURO_ARQUIVO)
print('LISTA_OURO_AVALIACAO:', LISTA_OURO_AVALIACAO_ARQUIVO)


## 4. Exportar tabelas-resumo


In [ ]:
out = OUTPUT_DIR / 'descritiva_bases'
out.mkdir(parents=True, exist_ok=True)

brasil_totais.to_csv(out / 'brasil_totais.csv', index=False)
brasil_miss_cpf.to_csv(out / 'brasil_completude_cpf.csv', index=False)
brasil_miss_censo.to_csv(out / 'brasil_completude_censo_pessoas.csv', index=False)
brasil_nome_censo.to_csv(out / 'brasil_completude_censo_nome.csv', index=False)
brasil_nome_cpf.to_csv(out / 'brasil_completude_cpf_nome.csv', index=False)
brasil_geo_cpf.to_csv(out / 'brasil_geo_cpf_por_uf.csv', index=False)
brasil_geo_censo.to_csv(out / 'brasil_geo_censo_por_uf.csv', index=False)
uf21_completude.to_csv(out / 'uf21_completude_limpo.csv', index=False)
uf21_tokens.to_csv(out / 'uf21_tokens_limpo.csv', index=False)
funil.to_csv(out / 'uf21_funil.csv', index=False)

print('Exportado em', out)
con.close()
